<a href="https://colab.research.google.com/github/G-Aswin/CudaLab/blob/main/CUDA_RUNNER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Details on the GPU

In [ ]:
!nvidia-smi

Fri Mar  6 16:02:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

NVCC version

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


Installing nvcc4jupyter. Below two commands need to be run and ready for %%cuda to work

In [ ]:
!pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpikra7px9".


Basic CUDA program (prints out the blockDim, blockIdx, global thread ID). %%cuda is a way to run cuda programs directly with the "run" button in the codeblock. Above extension needs to be loaded for this to work.

In [ ]:
%%cuda

#include <stdio.h>

// kernel function that runs on the GPU hardware
__global__ void simpleKernel() {
    int blockidx = blockIdx.x, blockdim = blockDim.x, threadidx = threadIdx.x;
    int tid = blockdim*blockidx + threadidx;
    printf("Hello world, from global tid : %d (blockIdx : %d, threadIdx : %d)\n", tid, blockidx, threadidx);
}

int main() {
    // Launching 1 block and 1 thread
    simpleKernel<<<2, 4>>>();

    // Wait for the GPU to finish its task before the CPU closes the program
    cudaDeviceSynchronize();

    return 0;
}

Hello world, from global tid : 0 (blockIdx : 0, threadIdx : 0)
Hello world, from global tid : 1 (blockIdx : 0, threadIdx : 1)
Hello world, from global tid : 2 (blockIdx : 0, threadIdx : 2)
Hello world, from global tid : 3 (blockIdx : 0, threadIdx : 3)
Hello world, from global tid : 4 (blockIdx : 1, threadIdx : 0)
Hello world, from global tid : 5 (blockIdx : 1, threadIdx : 1)
Hello world, from global tid : 6 (blockIdx : 1, threadIdx : 2)
Hello world, from global tid : 7 (blockIdx : 1, threadIdx : 3)



Another way to run cuda is to write into a cuda file, compile and run. This does not require any extensions.

In [7]:
%%writefile helloworld.cu
#include <stdio.h>
#include <cuda.h>
#include <cuda_runtime.h>

// kernel function that runs on the GPU hardware
__global__ void simpleKernel() {
    int blockidx = blockIdx.x, blockdim = blockDim.x, threadidx = threadIdx.x;
    int tid = blockdim*blockidx + threadidx;
    printf("Hello world, from global tid : %d (blockIdx : %d, threadIdx : %d)\n", tid, blockidx, threadidx);
}

int main() {
    // Launching 1 block and 1 thread
    simpleKernel<<<2, 4>>>();

    // Wait for the GPU to finish its task before the CPU closes the program
    cudaDeviceSynchronize();

    return 0;
}

Writing helloworld.cu


In [ ]:
!nvcc helloworld.cu -o helloworld -Wno-deprecated-gpu-targets
!./helloworld

Hello world, from global tid : 0 (blockIdx : 0, threadIdx : 0)
Hello world, from global tid : 1 (blockIdx : 0, threadIdx : 1)
Hello world, from global tid : 2 (blockIdx : 0, threadIdx : 2)
Hello world, from global tid : 3 (blockIdx : 0, threadIdx : 3)
Hello world, from global tid : 4 (blockIdx : 1, threadIdx : 0)
Hello world, from global tid : 5 (blockIdx : 1, threadIdx : 1)
Hello world, from global tid : 6 (blockIdx : 1, threadIdx : 2)
Hello world, from global tid : 7 (blockIdx : 1, threadIdx : 3)


Vector addition (with gpu memory allocation, loads and stores)

In [6]:
%%writefile vectorAddition.cu

#include <stdio.h>
#include <cuda.h>
#include <cuda_runtime.h>


// CUDA kernel for vector addition
__global__ void vectorAdd(float *A, float *B, float *C, int N) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < N) {
        C[i] = A[i] + B[i];
    }
}


int main()
{
    int N = 256; // Size of the vectors

    // Declare and allocate memory for host vectors h_A, h_B, and h_C
    float h_A[N], h_B[N], h_C[N];

    // Initialize the host vectors h_A and h_B with some values
    for (int i = 0; i < N; i++) {
        h_A[i] = i;
        h_B[i] = i*2;
    }

    // Allocate memory using cudaMalloc
    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, sizeof(float) * N);
    cudaMalloc((void**)&d_B, sizeof(float) * N);
    cudaMalloc((void**)&d_C, sizeof(float) * N);

    // Move data into this allocated space using cudaMemcpy
    cudaMemcpy(d_A, h_A, sizeof(float) * N, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, sizeof(float) * N, cudaMemcpyHostToDevice);

    // Invoke the vector addition kernel
    vectorAdd<<<(N + 255) / 256, 256>>>(d_A, d_B, d_C, N);

    // Copy back the data from device to host using cudaMemcpy
    cudaMemcpy(h_C, d_C, sizeof(float) * N, cudaMemcpyDeviceToHost);

    // Print the results
    for (int i = 0; i < N; i++) {
        printf("h_C[%d] = %f\n", i, h_C[i]);
    }

    // Free the device memory allocation using cudaFree
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

Overwriting vectorAddition.cu


In [7]:
!nvcc vectorAddition.cu -o vectorAddition -Wno-deprecated-gpu-targets
!./vectorAddition

h_C[0] = 0.000000
h_C[1] = 3.000000
h_C[2] = 6.000000
h_C[3] = 9.000000
h_C[4] = 12.000000
h_C[5] = 15.000000
h_C[6] = 18.000000
h_C[7] = 21.000000
h_C[8] = 24.000000
h_C[9] = 27.000000
h_C[10] = 30.000000
h_C[11] = 33.000000
h_C[12] = 36.000000
h_C[13] = 39.000000
h_C[14] = 42.000000
h_C[15] = 45.000000
h_C[16] = 48.000000
h_C[17] = 51.000000
h_C[18] = 54.000000
h_C[19] = 57.000000
h_C[20] = 60.000000
h_C[21] = 63.000000
h_C[22] = 66.000000
h_C[23] = 69.000000
h_C[24] = 72.000000
h_C[25] = 75.000000
h_C[26] = 78.000000
h_C[27] = 81.000000
h_C[28] = 84.000000
h_C[29] = 87.000000
h_C[30] = 90.000000
h_C[31] = 93.000000
h_C[32] = 96.000000
h_C[33] = 99.000000
h_C[34] = 102.000000
h_C[35] = 105.000000
h_C[36] = 108.000000
h_C[37] = 111.000000
h_C[38] = 114.000000
h_C[39] = 117.000000
h_C[40] = 120.000000
h_C[41] = 123.000000
h_C[42] = 126.000000
h_C[43] = 129.000000
h_C[44] = 132.000000
h_C[45] = 135.000000
h_C[46] = 138.000000
h_C[47] = 141.000000
h_C[48] = 144.000000
h_C[49] = 147.00000